# 概要
実績シートをSalesforceへインポートする

1.   スプレッドシートのURLを入力
2.   プログラムを実施（Ctrl + F9）

A〜AD列はSalesforceへ反映しません。B列はレコードIDの照合に使い、反映はAE列以降です。




In [ ]:
!pip install simple-salesforce gspread pandas
!pip install pyjwt cryptography requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 191.7/191.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 3.2 MB/s eta 0:00:00


In [ ]:
from google.colab import auth
auth.authenticate_user()

import re
import json
import time
import gspread
import google.auth
import requests
try:
    import jwt
except ImportError as e:
    raise RuntimeError("PyJWT が必要です。Colab で `!pip install pyjwt[crypto]` を実行してください。") from e
from google.colab import userdata
from datetime import datetime, timedelta

"""
要件:
- Google Colab で実行
- API 系情報は既存の main.py を参考。main.py は編集しない
- 読み込むのは「実績シート」
- B 列は稼働オブジェクトのレコード ID、1 行目は項目 API 名
- インポート対象は B 列（レコード ID）が空欄でない行すべて
- A〜AD列の値は Salesforce へ反映しない。反映は AE 列以降
"""

# ===== ロギング =====
def log(message: str) -> None:
    ts = datetime.now().strftime('%H:%M:%S')
    print(f"[{ts}] {message}")

# ===== 入力: スプレッドシート URL（/d/<ID>/ 形式 or 直接 ID） =====
# ラベル名を設定。掘起カードを作成する年月を指定
SOURCE_SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1U8xYZk_uWmfSWRPRBeHub3r9_MudqbfMXofxgvZsOQE/edit?gid=0#gid=0"  # @param {"type":"string"}
TARGET_SHEET_TITLE = "実績シート"

# ===== Google 認証（Sheets） =====
scopes = [
    "https://www.googleapis.com/auth/spreadsheets",
]
creds, _ = google.auth.default(scopes=scopes)
gc = gspread.authorize(creds)

# ===== スプレッドシートを開く =====
if not SOURCE_SPREADSHEET_URL or SOURCE_SPREADSHEET_URL == "PUT_YOUR_SPREADSHEET_URL_HERE":
    raise RuntimeError("SOURCE_SPREADSHEET_URL に対象スプレッドシートのURLまたはIDを設定してください（main.py の出力 new_url を貼り付け推奨）。")

max_attempts = 6
last_err = None
for attempt in range(max_attempts):
    try:
        if isinstance(SOURCE_SPREADSHEET_URL, str) and SOURCE_SPREADSHEET_URL.startswith("http"):
            log("スプレッドシートをURLで開きます …")
            sh = gc.open_by_url(SOURCE_SPREADSHEET_URL)
        else:
            # URLでない場合はIDとみなす
            log("スプレッドシートをIDで開きます …")
            sh = gc.open_by_key(SOURCE_SPREADSHEET_URL)
        break
    except gspread.exceptions.SpreadsheetNotFound as e:
        raise RuntimeError("スプレッドシートが見つからないか、アクセス権がありません。URL/IDを確認し、Colabで認証したGoogleアカウントに共有してください。") from e
    except gspread.exceptions.APIError as e:
        last_err = e
        if attempt == max_attempts - 1:
            raise
        time.sleep(min(60, 2 ** attempt))

spreadsheet_title = getattr(sh, 'title', '(no title)')
log(f"スプレッドシートを開きました: {spreadsheet_title}")
ws = sh.worksheet(TARGET_SHEET_TITLE)
log(f"ワークシート取得: {TARGET_SHEET_TITLE}")

# ===== シートからヘッダとデータを取得 =====
# 仕様: ヘッダは 1 行目、B 列以降。データは 2 行目以降。
# インポート対象は B 列（レコード ID）が空欄でない行すべて。
# A〜AD列はレコードIDの照合以外、Salesforceへ反映しない。
start_col = 2  # B 列
skip_import_through_col = 30  # AD 列
header_row = 1
data_start_row = 2

# ヘッダ取得（B1:末尾）
header_values = ws.row_values(header_row)
if len(header_values) < start_col:
    raise RuntimeError("B列以降にヘッダ(API名)が見つかりません")
headers_api_names = header_values[start_col - 1:]
# シートの出力項目はそのまま。Salesforce への取り込み先だけ付け替える
IMPORT_FIELD_REMAP = {
    "ownership__c": "zissekishiensyubetsu__c",
    "ryunyumototesuryo__c": "zissekishienhi__c",
}
# シート作成時に税込金額を税抜（÷1.1）へ置き換えている列。
# 税区分は税込のまま戻すと、インボイス負担費と粗利で消費税を二重に引く。
SHEET_TAX_EXCLUSIVE_DISPLAY_FIELDS = {"uekingaku__c", "shitakingaku__c"}
headers_api_names = [IMPORT_FIELD_REMAP.get(name, name) for name in headers_api_names]
for _src_name, _dest_name in IMPORT_FIELD_REMAP.items():
    log(f"取り込み先の付け替え: {_src_name} → {_dest_name}")
log(f"実績シート ヘッダ列数: {len(headers_api_names)}")
log("A〜AD列はSalesforceへ反映しません。反映はAE列以降です。")

# 全データ範囲をまとめて取得（B2:最終行, 最終列はヘッダ長で決定）
last_row = ws.row_count
last_col = start_col + max(len(headers_api_names), 1) - 1
data_range = f"{gspread.utils.rowcol_to_a1(data_start_row, start_col)}:{gspread.utils.rowcol_to_a1(last_row, last_col)}"
values = ws.get(data_range, value_render_option='UNFORMATTED_VALUE')
log(f"実績シート データ取得: 範囲 {data_range} / 行数 {len(values)}")


def _is_blank_cell(value) -> bool:
    if value is None:
        return True
    text = str(value).strip()
    return text == ""


# B列（レコード ID）がある行だけを対象に、{api_name: value, ...} へマッピング
records = []
skipped_no_id = 0
for idx, row in enumerate(values, start=data_start_row):
    if not row or (len(row) == 0):
        continue

    # B列の ID は最初のキー（headers_api_names[0]）に入っている想定
    record_id = row[0] if len(row) > 0 else None
    if _is_blank_cell(record_id):
        skipped_no_id += 1
        continue

    mapped = {}
    for j, api_name in enumerate(headers_api_names):
        # A〜AD列は反映しない。B列は上でレコードIDとして使っている
        if start_col + j <= skip_import_through_col:
            continue
        # row[j] は存在しない場合があるので安全に取得
        value = row[j] if j < len(row) else None
        mapped[api_name] = value
    records.append({"Id": record_id, "fields": mapped})
log(f"実績シート レコード数: {len(records)} / ID空欄スキップ: {skipped_no_id}")

# ===== Salesforce 認証（JWT Bearer フロー） =====
SF_CLIENT_ID = userdata.get('SF_CLIENT_ID')
SF_USERNAME = userdata.get('SF_USERNAME')
SF_PRIVATE_KEY = userdata.get('SF_PRIVATE_KEY')

if not SF_CLIENT_ID or not SF_USERNAME or not SF_PRIVATE_KEY:
    raise RuntimeError('Salesforce シークレット (SF_CLIENT_ID / SF_USERNAME / SF_PRIVATE_KEY) が見つかりません。')

SF_TOKEN_URL = 'https://login.salesforce.com/services/oauth2/token'


def _normalize_private_key(raw_key: str) -> str:
    return str(raw_key).strip().replace("\\n", "\n")


def sf_auth():
    now = int(time.time())
    claims = {
        'iss': SF_CLIENT_ID,
        'sub': SF_USERNAME,
        'aud': 'https://login.salesforce.com',
        'exp': now + 180,
    }
    assertion = jwt.encode(claims, _normalize_private_key(SF_PRIVATE_KEY), algorithm='RS256')
    token_resp = requests.post(
        SF_TOKEN_URL,
        data={
            'grant_type': 'urn:ietf:params:oauth:grant-type:jwt-bearer',
            'assertion': assertion,
        },
        timeout=60,
    )
    token_resp.raise_for_status()
    token_json = token_resp.json()
    return token_json['access_token'], token_json['instance_url']


access_token, instance_url = sf_auth()
log(f"Salesforce 認証完了: {instance_url}")

# 稼働オブジェクト API 名（main.py と同様の優先順位）
try:
    KADO_OBJECT_API = userdata.get('Salesforce_kado_object')
except Exception:
    KADO_OBJECT_API = None
if not KADO_OBJECT_API:
    KADO_OBJECT_API = 'kado__c'

headers = {
    'Authorization': f'Bearer {access_token}',
    'Content-Type': 'application/json',
}

# ===== 対象オブジェクトの項目メタを取得（型に合わせて正規化するため） =====
describe_url = f"{instance_url}/services/data/v57.0/sobjects/{KADO_OBJECT_API}/describe"
desc_resp = requests.get(describe_url, headers=headers, timeout=60)
desc_resp.raise_for_status()
describe = desc_resp.json()
log(f"稼働オブジェクトメタ取得: {KADO_OBJECT_API} / 項目数 {len(describe.get('fields', []))}")

field_name_to_meta = {}
for f in describe.get("fields", []):
    # name: API名, type: 項目型, updateable: 更新可能か
    field_name_to_meta[f.get("name")] = {
        "type": f.get("type"),
        "updateable": f.get("updateable", True),
        "createable": f.get("createable", True),
    }

# スプレッドシート名先頭の YYYYMM（例: 202605_実績シート）から年を推定
_title_year_match = re.match(r"^(\d{4})\d{2}", spreadsheet_title or "")
INFERRED_DEFAULT_YEAR = int(_title_year_match.group(1)) if _title_year_match else datetime.now().year

_SHEETS_EPOCH = datetime(1899, 12, 30)


def _try_parse_with_patterns(text: str, patterns):
    for p in patterns:
        try:
            return datetime.strptime(text, p)
        except Exception:
            pass
    return None


def _sheets_serial_to_datetime(serial: float) -> datetime:
    """Google スプレッドシートのシリアル値を datetime に変換する。"""
    days = int(serial)
    fraction = serial - days
    dt = _SHEETS_EPOCH + timedelta(days=days)
    if fraction:
        dt += timedelta(seconds=round(fraction * 86400))
    return dt


def _parse_date_text(text: str, default_year: int | None = None) -> datetime | None:
    """日付文字列を解析する。年なし M/D 形式にも対応。"""
    year = default_year or INFERRED_DEFAULT_YEAR
    dt_obj = _try_parse_with_patterns(text, [
        "%Y/%m/%d",
        "%Y-%m-%d",
        "%Y.%m.%d",
        "%Y%m%d",
        "%m/%d/%Y",
        "%m/%d/%y",
        "%Y/%m/%d %H:%M",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
    ])
    if dt_obj is not None:
        return dt_obj

    m = re.fullmatch(r"(\d{1,2})/(\d{1,2})", text)
    if m:
        try:
            return datetime(year, int(m.group(1)), int(m.group(2)))
        except ValueError:
            return None
    return None


def _parse_datetime_text(text: str) -> datetime | None:
    dt_obj = _try_parse_with_patterns(text, [
        "%Y/%m/%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y/%m/%d %H:%M",
        "%Y-%m-%d %H:%M",
        "%Y/%m/%d",
        "%Y-%m-%d",
        "%Y-%m-%dT%H:%M:%S.%fZ",
        "%Y-%m-%dT%H:%M:%S.%f%z",
        "%Y-%m-%dT%H:%M:%SZ",
        "%Y-%m-%dT%H:%M:%S%z",
    ])
    if dt_obj is not None:
        return dt_obj
    return _parse_date_text(text)


def normalize_value_by_type(value_raw, field_meta):
    if value_raw is None:
        return None

    field_type = (field_meta or {}).get("type")

    # スプレッドシートの日付シリアル値（UNFORMATTED_VALUE 取得時）
    if field_type in ("date", "datetime") and isinstance(value_raw, (int, float)):
        dt_obj = _sheets_serial_to_datetime(float(value_raw))
        if field_type == "date":
            return dt_obj.strftime("%Y-%m-%d")
        if dt_obj.hour == 0 and dt_obj.minute == 0 and dt_obj.second == 0:
            return dt_obj.strftime("%Y-%m-%dT00:00:00Z")
        return dt_obj.strftime("%Y-%m-%dT%H:%M:%SZ")

    text = str(value_raw).strip()
    if text == "":
        return None
    # ダッシュのみは全型で未設定扱い（送信除外）
    if text in ("-", "ー", "—", "―", "－"):
        return None

    if field_type == "date":
        dt_obj = _parse_date_text(text)
        if dt_obj is None:
            return None
        return dt_obj.strftime("%Y-%m-%d")

    if field_type == "datetime":
        dt_obj = _parse_datetime_text(text)
        if dt_obj is None:
            return None
        if dt_obj.hour == 0 and dt_obj.minute == 0 and dt_obj.second == 0 and (" ") not in text and "T" not in text:
            return dt_obj.strftime("%Y-%m-%dT00:00:00Z")
        return dt_obj.strftime("%Y-%m-%dT%H:%M:%SZ")

    if field_type in ("double", "currency", "percent"):
        # ダッシュのみは空値扱い
        if text in ("-", "ー", "—", "―", "－"):
            return None
        # 全角->半角変換（数字と記号の一部）
        trans = str.maketrans({"０":"0","１":"1","２":"2","３":"3","４":"4","５":"5","６":"6","７":"7","８":"8","９":"9","．":".","－":"-","，":","})
        text_hw = text.translate(trans)
        # 括弧付き負数 (1,234) → -1234
        m_neg = re.fullmatch(r"\((.+)\)", text_hw)
        if m_neg:
            text_hw = f"-{m_neg.group(1)}"
        # 桁区切り/通貨/空白/% を除去（% は数値として解釈）
        cleaned = (
            text_hw.replace(",", "")
            .replace("¥", "")
            .replace("￥", "")
            .replace("円", "")
            .replace("%", "")
            .strip()
        )
        try:
            return float(cleaned)
        except Exception:
            return text

    if field_type in ("int", "integer", "long"):
        if text in ("-", "ー", "—", "―", "－"):
            return None
        trans = str.maketrans({"０":"0","１":"1","２":"2","３":"3","４":"4","５":"5","６":"6","７":"7","８":"8","９":"9","．":".","－":"-","，":","})
        text_hw = text.translate(trans)
        m_neg = re.fullmatch(r"\((.+)\)", text_hw)
        if m_neg:
            text_hw = f"-{m_neg.group(1)}"
        cleaned = text_hw.replace(",", "")
        try:
            return int(float(cleaned))
        except Exception:
            return text

    if field_type == "boolean":
        lowered = text.lower()
        if lowered in ("true", "1", "yes", "y", "on", "はい", "有", "○"):
            return True
        if lowered in ("false", "0", "no", "n", "off", "いいえ", "無", "×"):
            return False
        return text

    if field_type == "multipicklist":
        # 想定区切り: ';'。全角読点などは置換
        normalized = text.replace("、", ";").replace("，", ";")
        return normalized

    # その他（string, picklist, email, phone, url, textarea, reference 等）はそのまま
    return text

# ===== レコード更新（PATCH）
# Headers の 1 列目（B列の API 名）は ID 用の列であるため、更新対象フィールドからは除外
if headers_api_names:
    id_field_api_name = headers_api_names[0]
else:
    id_field_api_name = None

updated_count = 0
failed = []
processed = 0
total = len(records)

for rec in records:
    record_id = rec["Id"]
    fields_all = rec["fields"].copy()
    # ID が入っている列を除外
    if id_field_api_name and id_field_api_name in fields_all:
        fields_all.pop(id_field_api_name, None)

    # 空値は送らない。型に合わせて正規化し、更新不可項目は送らない
    payload = {}
    for k, v in fields_all.items():
        if v in (None, ""):
            continue
        meta = field_name_to_meta.get(k)
        if k in SHEET_TAX_EXCLUSIVE_DISPLAY_FIELDS:
            continue
        if meta is not None and not meta.get("updateable", True):
            continue
        normalized = normalize_value_by_type(v, meta)
        if normalized in (None, ""):
            continue
        payload[k] = normalized
    if not payload:
        continue

    url = f"{instance_url}/services/data/v57.0/sobjects/{KADO_OBJECT_API}/{record_id}"
    try:
        resp = requests.patch(url, headers=headers, data=json.dumps(payload), timeout=60)
        if resp.status_code in (204, 200):
            updated_count += 1
        else:
            failed.append((record_id, resp.status_code, resp.text))
    except Exception as e:
        failed.append((record_id, "EXC", str(e)))
    finally:
        processed += 1
        if processed % 20 == 0 or processed == total:
            log(f"実績シート 更新進捗: {processed}/{total} (成功 {updated_count}, 失敗 {len(failed)})")

# ===== 結果のサマリ出力 =====
log(f"更新成功: {updated_count} 件")
if failed:
    log(f"更新失敗: {len(failed)} 件")
    # 先頭数件のみ表示
    for item in failed[:10]:
        print("FAIL", item)

# ===== 口座マスタのインポート（A列: 口座オブジェクトのレコードID, 1行目: API名） =====
try:
    ws_master = sh.worksheet("口座マスタ")
except Exception as e:
    raise RuntimeError("シート『口座マスタ』が見つかりません。main.py で作成された同一スプレッドシートを指定してください。") from e

# ヘッダ取得（A1: 最終列）
header_master_raw = ws_master.row_values(1)
if not header_master_raw:
    raise RuntimeError("口座マスタ: 1行目のヘッダ(API名)が空です。")

# ヘッダのAPI名調整: CUST_NAME/NAME → Name
def _map_bank_header(name):
    if not isinstance(name, str):
        return name
    n = name.strip()
    if n.upper() in ("CUST_NAME", "NAME"):
        return "Name"
    return n

header_master = [_map_bank_header(h) for h in header_master_raw]
log(f"口座マスタ ヘッダ列数: {len(header_master)}")

id_field_api_name_master = header_master[0]

# 値の取得（A2: 最終行 × ヘッダ列数）
last_row_master = ws_master.row_count
last_col_master = 1 + max(len(header_master), 1) - 1
data_range_master = f"{gspread.utils.rowcol_to_a1(2, 1)}:{gspread.utils.rowcol_to_a1(last_row_master, last_col_master)}"
values_master = ws_master.get(data_range_master, value_render_option='UNFORMATTED_VALUE')
log(f"口座マスタ データ取得: 範囲 {data_range_master} / 行数 {len(values_master)}")

# 口座オブジェクト API 名（ユーザーデータ優先、未設定時は 'bankaccount__c'）
try:
    BANK_OBJECT_API = userdata.get('Salesforce_bankaccount_object')
except Exception:
    BANK_OBJECT_API = None
if not BANK_OBJECT_API:
    BANK_OBJECT_API = 'bankaccount__c'

# 項目メタ（口座オブジェクト）
describe_url_bank = f"{instance_url}/services/data/v57.0/sobjects/{BANK_OBJECT_API}/describe"
desc_bank_resp = requests.get(describe_url_bank, headers=headers, timeout=60)
desc_bank_resp.raise_for_status()
describe_bank = desc_bank_resp.json()
log(f"口座オブジェクトメタ取得: {BANK_OBJECT_API} / 項目数 {len(describe_bank.get('fields', []))}")

bank_field_meta = {}
for f in describe_bank.get("fields", []):
    bank_field_meta[f.get("name")] = {
        "type": f.get("type"),
        "updateable": f.get("updateable", True),
        "createable": f.get("createable", True),
    }

def chunked(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

# 1) 行ごとの正規化・仕分け
update_candidates = []  # {idx, rec_id, payload_update}
create_candidates = []  # {idx, payload_create}

for idx, row in enumerate(values_master, start=2):
    if not row or all((c in (None, "")) for c in row):
        continue

    rec_id = row[0] if len(row) > 0 and row[0] not in (None, "") else None
    fields_map = {}
    for j, api_name in enumerate(header_master):
        val = row[j] if j < len(row) else None
        fields_map[api_name] = val
    if id_field_api_name_master in fields_map:
        fields_map.pop(id_field_api_name_master, None)

    # 正規化
    normalized = {}
    for k, v in fields_map.items():
        if v in (None, ""):
            continue
        meta = bank_field_meta.get(k)
        normalized_val = normalize_value_by_type(v, meta)
        if normalized_val in (None, ""):
            continue
        normalized[k] = normalized_val

    if rec_id:
        # updateable でフィルタ
        payload_update = {}
        for k, v in normalized.items():
            meta = bank_field_meta.get(k)
            if meta is not None and not meta.get("updateable", True):
                continue
            payload_update[k] = v
        if payload_update:
            update_candidates.append({"idx": idx, "rec_id": rec_id, "payload": payload_update})
    else:
        # createable でフィルタ
        payload_create = {}
        for k, v in normalized.items():
            meta = bank_field_meta.get(k)
            if meta is not None and not meta.get("createable", True):
                continue
            payload_create[k] = v
        if payload_create:
            create_candidates.append({"idx": idx, "payload": payload_create})

log(f"口座マスタ 振り分け: 更新候補 {len(update_candidates)} 件, 作成候補 {len(create_candidates)} 件")

# 2) 既存値を一括取得して差分判定
def fetch_existing_records(ids, field_list):
    existing = {}
    if not ids or not field_list:
        return existing
    # Id は常に取得する（マッピングに使用）
    field_set_local = set(field_list)
    field_set_local.add('Id')
    fields_param = ",".join(sorted(field_set_local))
    for batch in chunked(ids, 200):
        url = f"{instance_url}/services/data/v57.0/composite/sobjects/{BANK_OBJECT_API}?ids={requests.utils.quote(','.join(batch))}&fields={requests.utils.quote(fields_param)}"
        resp = requests.get(url, headers=headers, timeout=60)
        if resp.status_code == 200:
            data = resp.json() or []
            # 応答が dict(results=...) または list の両方に対応
            if isinstance(data, dict):
                results = data.get("results", [])
                for r in results:
                    rid = r.get("id") or r.get("Id")
                    rec = r.get("record") or {}
                    if rid:
                        existing[rid] = rec
            elif isinstance(data, list):
                for item in data:
                    rid = item.get("Id") or item.get("id")
                    if rid:
                        existing[rid] = item
        else:
            log(f"WARN: 既存値一括取得 失敗 {resp.status_code}")
    return existing

updates_to_send = []
if update_candidates:
    # フィールド集合
    field_set = set()
    for u in update_candidates:
        field_set.update(u["payload"].keys())
    field_list = sorted(field_set)
    existing_map = fetch_existing_records([u["rec_id"] for u in update_candidates], field_list)

    for u in update_candidates:
        rec_id = u["rec_id"]
        payload = u["payload"]
        current = existing_map.get(rec_id, {})
        all_same = True
        for k, new_val in payload.items():
            meta = bank_field_meta.get(k)
            old_val_norm = normalize_value_by_type(current.get(k), meta)
            if old_val_norm != new_val:
                all_same = False
                break
        if not all_same:
            updates_to_send.append(u)

log(f"口座マスタ 差分あり更新: {len(updates_to_send)} 件")

# 3) 一括アップサート（更新・作成）
def send_composite_records(records_payload, http_method: str = 'POST', meta_records=None):
    failed_local = []
    success_map = {}  # key(meta) -> id or True
    if meta_records is None:
        meta_records = [None] * len(records_payload)
    for batch, meta_batch in zip(chunked(records_payload, 200), chunked(meta_records, 200)):
        body = {
            "allOrNone": False,
            "records": batch,
        }
        url = f"{instance_url}/services/data/v57.0/composite/sobjects"
        if http_method.upper() == 'PATCH':
            resp = requests.patch(url, headers=headers, data=json.dumps(body), timeout=120)
        else:
            resp = requests.post(url, headers=headers, data=json.dumps(body), timeout=120)
        if resp.status_code in (200, 201):
            res = resp.json() or []
            # 応答は list 形式（各要素に referenceId/success/id/errors）
            if isinstance(res, list):
                items = res
            else:
                # 念のため dict 形式(results=...) も許容
                items = res.get("results", []) if isinstance(res, dict) else []
            for i, r in enumerate(items):
                meta_info = meta_batch[i] if i < len(meta_batch) else None
                key = None
                if isinstance(meta_info, dict):
                    # シート行番号をキーにする
                    key = meta_info.get('idx')
                ref = (r.get("referenceId") if isinstance(r, dict) else None) or key or ""
                ok = isinstance(r, dict) and r.get("success") is True
                if ok:
                    sid = r.get("id") if isinstance(r, dict) else None
                    success_map[ref] = sid or True
                else:
                    errs = r.get("errors") if isinstance(r, dict) else None
                    failed_local.append((ref or "(unknown)", "ERR", errs))
        else:
            failed_local.append(("batch", resp.status_code, resp.text))
    return success_map, failed_local

# 構築: 更新
records_payload_updates = []
records_payload_updates_meta = []
for u in updates_to_send:
    rec = {"attributes": {"type": BANK_OBJECT_API}, "Id": u["rec_id"]}
    rec.update(u["payload"])
    records_payload_updates.append(rec)
    records_payload_updates_meta.append({"idx": u['idx'], "rec_id": u['rec_id']})

# 構築: 作成
records_payload_creates = []
records_payload_creates_meta = []
for c in create_candidates:
    rec = {"attributes": {"type": BANK_OBJECT_API}}
    rec.update(c["payload"])
    records_payload_creates.append(rec)
    records_payload_creates_meta.append({"idx": c['idx']})

created_count_master = 0
updated_count_master = 0
failed_master = []

# 送信: 更新（PATCH）
if records_payload_updates:
    log(f"口座マスタ 更新バッチ送信: {len(records_payload_updates)} 件")
    success_u, failed_u = send_composite_records(records_payload_updates, 'PATCH', records_payload_updates_meta)
    failed_master.extend(failed_u)
    updated_count_master = len(success_u)

# 送信: 作成（POST）
created_id_map = {}
if records_payload_creates:
    log(f"口座マスタ 作成バッチ送信: {len(records_payload_creates)} 件")
    success_ids, failed_c = send_composite_records(records_payload_creates, 'POST', records_payload_creates_meta)
    failed_master.extend(failed_c)
    created_count_master = len(success_ids)
    created_id_map.update({k: (v if isinstance(v, str) else None) for k, v in success_ids.items()})

# A列へID書き戻し（作成のみ）
for c in create_candidates:
    new_id = created_id_map.get(c['idx'])
    if new_id:
        ws_master.update_cell(c['idx'], 1, new_id)

log(f"口座マスタ 更新: {updated_count_master} 件, 作成: {created_count_master} 件")
if failed_master:
    log(f"口座マスタ 失敗: {len(failed_master)} 件")
    for item in failed_master[:10]:
        print("MASTER_FAIL", item)

pass


[01:33:01] スプレッドシートをURLで開きます …
[01:33:02] スプレッドシートを開きました: 202609_実績シート（インボイス対応）
[01:33:02] ワークシート取得: 実績シート
[01:33:02] 実績シート ヘッダ列数: 52
[01:33:03] 実績シート データ取得: 範囲 B6:BA1274 / 行数 1269
[01:33:03] 実績シート レコード数: 181
[01:33:05] Salesforce 認証完了: https://rosca.my.salesforce.com
[01:33:06] 稼働オブジェクトメタ取得: kado__c / 項目数 178
[01:33:16] 実績シート 更新進捗: 20/181 (成功 20, 失敗 0)
[01:33:24] 実績シート 更新進捗: 40/181 (成功 40, 失敗 0)
[01:33:32] 実績シート 更新進捗: 60/181 (成功 60, 失敗 0)
[01:33:41] 実績シート 更新進捗: 80/181 (成功 80, 失敗 0)
[01:33:49] 実績シート 更新進捗: 100/181 (成功 100, 失敗 0)
[01:33:56] 実績シート 更新進捗: 120/181 (成功 120, 失敗 0)
[01:34:04] 実績シート 更新進捗: 140/181 (成功 140, 失敗 0)
[01:34:13] 実績シート 更新進捗: 160/181 (成功 160, 失敗 0)
[01:34:20] 実績シート 更新進捗: 180/181 (成功 180, 失敗 0)
[01:34:20] 実績シート 更新進捗: 181/181 (成功 181, 失敗 0)
[01:34:20] 更新成功: 181 件
[01:34:21] 口座マスタ ヘッダ列数: 7
[01:34:21] 口座マスタ データ取得: 範囲 A2:G837 / 行数 836
[01:34:22] 口座オブジェクトメタ取得: bankaccount__c / 項目数 16
[01:34:22] 口座マスタ 振り分け: 更新候補 836 件, 作成候補 0 件
[01:34:23] 口座マスタ 差分あり更新: 0 件
[01:34:23] 口座マスタ 更新: 